In [1]:
import pyecharts
pyecharts.__version__

'2.1.0'

In [2]:
from pyecharts.globals import CurrentConfig, NotebookType

CurrentConfig.ONLINE_HOST = "https://cdn.bootcdn.net/ajax/libs/echarts/5.5.0/"# 或者直接修改库的default host
CurrentConfig.NOTEBOOK_TYPE = NotebookType.JUPYTER_NOTEBOOK

In [3]:
from IPython.display import display, HTML
import html

_echarts_loaded = False

def _ensure_echarts_loaded():
    """确保 ECharts 已加载到页面"""
    global _echarts_loaded
    if not _echarts_loaded:
        display(HTML(f"""
        <script>
            if (typeof window.echarts === 'undefined') {{
                var script = document.createElement('script');
                script.src = '{CurrentConfig.ONLINE_HOST}echarts.min.js';
                document.head.appendChild(script);
                window.__echarts_loading = true;
            }}
        </script>
        """))
        _echarts_loaded = True
        
def render_in_iframe(chart):
    """HTML沙箱化：iframe 中渲染 pyecharts，绕过浏览器“Tracking Prevention blocked access to storage for <URL>.”"""
    _ensure_echarts_loaded()
    # 获取图表的 HTML 片段
    html_content = chart.render_embed()
    
    # HTML 转义
    escaped_html = html.escape(html_content)
    
    # 动态获取图表的宽高
    width = getattr(chart, 'width', '100%')
    height = getattr(chart, 'height', '500px')
    
    # 增加 scrolling 和 sandbox 属性提升体验和安全性
    iframe_html = f"""
    <iframe srcdoc="{escaped_html}" 
            width="{width}" 
            height="{height}" 
            frameborder="0"
            scrolling="no"
            sandbox="allow-scripts">
    </iframe>
    """
    return HTML(iframe_html)

# 3-1例 单数据柱状图绘制

In [4]:
from pyecharts.charts import Bar
from pyecharts import options as opts

In [5]:
bar = Bar()
bar.add_xaxis(["衬衫", "毛衣", "领带", "裤子", "风衣", "高跟鞋", "袜子"])
bar.add_yaxis("商家A", [114, 55, 27, 101, 125, 27, 105])
bar.set_global_opts(
    #标题配置项
    title_opts = opts.TitleOpts(title = "货品销售情况",subtitle  =  "A公司"))

#bar.load_javascript()
#bar.render_notebook()
render_in_iframe(bar)

In [6]:
!jupyter lab --version

4.5.6


# 3-1例 利用Bar绘制柱状图

In [7]:
# V1 版本开始支持链式调用
bar = ( 
    Bar()
    .add_xaxis(["衬衫", "毛衣", "领带", "裤子", "风衣","高跟鞋", "袜子"])
    .add_yaxis("商家A", [114, 55, 27, 101, 125, 27, 105])
    .add_yaxis("商家B", [57, 134, 137, 129, 145, 60, 49])
    .set_global_opts(title_opts = opts.TitleOpts(title = "某商场销售情况")) 
)


render_in_iframe(bar)

# 3-1例 并列柱状图绘制（多数据系列柱状图）

In [8]:
bar = Bar(init_opts=opts.InitOpts(
            #设置动画
            animation_opts=opts.AnimationOpts(animation_delay=1000, animation_easing="elasticOut"),
            #设置宽度、高度
            width='900px',
            height='500px'))
bar.add_xaxis(["衬衫", "毛衣", "领带", "裤子", "风衣", "高跟鞋", "袜子"])
bar.add_yaxis("商家A", [114, 55, 27, 101, 125, 27, 105])
bar.add_yaxis("商家B", [57, 134, 137, 129, 145, 60, 49])
bar.set_global_opts(
    #标题配置项
    title_opts = opts.TitleOpts(title = "货品销售情况",subtitle  =  "A和B公司"))


render_in_iframe(bar)
#bar.render("11.html")


# 3-1例 条形图

In [9]:
bar =(
    Bar()
    .add_xaxis(["衬衫", "毛衣", "领带", "裤子", "风衣", "高跟鞋", "袜子"])
    .add_yaxis("商家A", [114, 55, 27, 101, 125, 27, 105])
    .add_yaxis("商家B", [57, 134, 137, 129, 145, 60, 49])
    .set_global_opts(
        #标题配置项
        title_opts=opts.TitleOpts(title="货品销售情况",subtitle = "A和B公司"),
        #工具箱配置项
        toolbox_opts = opts.ToolboxOpts(is_show = True))
    
    .set_series_opts(label_opts=opts.LabelOpts(position = "right"))  #标签配置项
    #翻转x、y轴
    .reversal_axis()
)
render_in_iframe(bar)


# 3-1 堆栈柱状图

In [10]:
from pyecharts.charts import Bar
from pyecharts import options as opts

bar =(
    Bar()
    .add_xaxis(["衬衫", "毛衣", "领带", "裤子", "风衣", "高跟鞋", "袜子"])
    .add_yaxis("商家A", [114, 55, 27, 101, 125, 27, 105],stack="stack1",bar_width=40)
    .add_yaxis("商家B", [57, 134, 137, 129, 145, 60, 49],stack="stack1",bar_width=40)
    .set_global_opts(
        #标题配置项
        title_opts = opts.TitleOpts(title = "货品销售情况",subtitle  =  "A和B公司"))
    #标签配置项:在右边显示标签
    .set_series_opts(label_opts=opts.LabelOpts(position = "right"))
)

render_in_iframe(bar)



# pyechart+notebook cell的html导出
!jupyter nbconvert --to html 3-1bar.ipynb